# 🎙️🎭 Server GABUNGAN F5-TTS (Flow Matching) + Talking Avatar (PippitLokal)

Satu Colab, satu URL. Endpoint `/tts` (voice clone F5-TTS speed=0.78) **dan** `/avatar` (video talking avatar) jalan bersama.

**Keunggulan F5-TTS:**
- Suara tenang, ritme natural (speed=0.78, nfe_step=64)
- VRAM hemat (~3.5 GB) — sangat stabil di GPU T4 tanpa risiko OOM
- Kloning akurat dari sampel audio 3–10 detik

**Cara pakai:** Runtime → Change runtime type → **T4 GPU** → Save, lalu **Runtime → Run all**.
Tunggu sampai muncul **URL SERVER** (…trycloudflare.com). Salin URL itu ke PippitLokal:
- kolom *URL server VoxCPM/F5-TTS* (voice clone), dan
- setting `avatarColabUrl` (pakai URL yang **sama**).


### 1) Install F5-TTS + engine avatar (SadTalker + LivePortrait)
Sekali per sesi. Sekitar ~3-5 menit.

In [ ]:
# Install F5-TTS + Faster-Whisper + clone repo + setup avatar
!pip -q install f5-tts faster-whisper soundfile torchaudio
!git clone -b feat/talking-avatar https://github.com/dionisius95/yt-short.git /content/yt-short 2>/dev/null || (cd /content/yt-short && git pull)
!bash /content/yt-short/colab/setup_colab.sh

### 2) Set environment engine avatar (T4-friendly)

In [ ]:
import os
os.environ.pop('PYTHONHASHSEED', None)
os.environ['SADTALKER_DIR']    = '/content/SadTalker'
os.environ['LIVEPORTRAIT_DIR'] = '/content/LivePortrait'
os.environ['IDLE_DRIVING']     = '/content/assets/idle_driving.mp4'
os.environ['AVATAR_MAX_SIDE']  = '256'   # T4 optimal & cepat
os.environ['AVATAR_FP16']      = '1'
os.environ['AVATAR_FPS']       = '25'
os.environ['AVATAR_WORKDIR']   = '/content/avatar_work'

### 3) Tulis server gabungan F5-TTS + Avatar

In [ ]:
%%writefile /content/pippit_server.py
# -*- coding: utf-8 -*-
"""
pippit_server.py - server GABUNGAN F5-TTS + Talking Avatar untuk PippitLokal.

Satu port, satu URL cloudflared:
  GET  /health         -> 'ok' (200 HANYA setelah model F5-TTS termuat)
  GET  /avatar/health  -> JSON status engine avatar (talk/idle/device)
  POST /tts  (atau /clone) -> audio/wav  (voice clone F5-TTS Flow Matching)
  POST /avatar (atau /lipsync) -> 202 JSON { job_id, status }  (ASYNC)
  GET  /avatar/result/<job_id> -> 202 (proses) | video/mp4 (selesai) | 5xx JSON (gagal)

Avatar render dijalankan di thread background supaya tiap request pendek dan
tidak pernah kena batas ~100s Cloudflare quick tunnel (HTTP 524). Fungsi render
diimpor langsung dari repo yt-short (colab/avatar_server.py).
"""
import os, io, sys, json, base64, tempfile, traceback, subprocess, threading, uuid, time
from pathlib import Path
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
import soundfile as sf
import torch

# Bersihkan environment dari variabel penyebab crash fatal Python
os.environ.pop('PYTHONHASHSEED', None)

# ---- Impor F5-TTS & Whisper ----
from f5_tts.api import F5TTS
from faster_whisper import WhisperModel

# ---- Impor fungsi render avatar dari repo ----
sys.path.insert(0, "/content/yt-short/colab")
from avatar_server import (
    _prep_image, _render_sadtalker, _render_liveportrait_idle,
    _make_silent_wav, _loop_to_duration, _sadtalker_available, _liveportrait_available,
    WORKDIR, DEFAULT_FPS,
)
try:
    from avatar_server import remove_background_video
except Exception:
    remove_background_video = None

PORT = int(os.environ.get("F5_PORT", os.environ.get("VOXCPM_PORT", "8081")))
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"[F5-TTS] Memuat model F5-TTS (device={DEVICE}) ...", flush=True)
try:
    try:
        F5_MODEL = F5TTS(model="F5TTS_Base", device=DEVICE)
    except TypeError:
        try:
            F5_MODEL = F5TTS(device=DEVICE)
        except TypeError:
            F5_MODEL = F5TTS(model_type="F5-TTS", device=DEVICE)
    print("[F5-TTS] Model siap 100% untuk voice cloning!", flush=True)
except Exception:
    print("[F5-TTS] GAGAL memuat model F5-TTS:", flush=True)
    traceback.print_exc()
    sys.exit(1)

# Lazy Whisper untuk auto-transcribe sampel suara
_WHISPER_MODEL = None
_WHISPER_LOCK = threading.Lock()

def _get_whisper():
    global _WHISPER_MODEL
    with _WHISPER_LOCK:
        if _WHISPER_MODEL is None:
            print("[Whisper] Memuat faster-whisper base untuk transkrip otomatis sampel suara...", flush=True)
            _WHISPER_MODEL = WhisperModel("base", device=DEVICE, compute_type="float16" if DEVICE == "cuda" else "int8")
        return _WHISPER_MODEL

def _transcribe_ref_audio(wav_path):
    try:
        model = _get_whisper()
        segments, _ = model.transcribe(wav_path, beam_size=3)
        text = " ".join([seg.text.strip() for seg in segments]).strip()
        print(f"[Whisper] Transkrip otomatis sampel: '{text}'", flush=True)
        return text
    except Exception as e:
        print(f"[Whisper] Peringatan: Transkrip otomatis gagal ({e}), lanjut dengan ref_text kosong.", flush=True)
        return ""


def _render_static_idle(img_path, out_dir, fps, duration):
    """Idle TENANG tanpa gerak wajah: potret statis + micro-zoom sangat halus."""
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    out = out_dir / "idle_static.mp4"
    dur = max(0.5, float(duration))
    nframes = max(1, int(round(dur * int(fps))))
    vf = (
        "scale=512:512:force_original_aspect_ratio=increase,crop=512:512,"
        "zoompan=z='min(1.0+0.0006*on,1.06)':d=%d:s=256x256:fps=%d,"
        "format=yuv420p" % (nframes, int(fps))
    )
    subprocess.run([
        "ffmpeg", "-y", "-loop", "1", "-i", str(img_path),
        "-t", "%.3f" % dur, "-r", str(int(fps)), "-vf", vf,
        "-c:v", "libx264", "-crf", "20", "-pix_fmt", "yuv420p", str(out),
    ], stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT, check=True)
    return out


def _render_avatar_to_file(data):
    """Render mp4 avatar dan balikan PATH file (tidak dihapus). Reuse avatar_server.py."""
    image_b64 = data.get("image_b64")
    audio_b64 = data.get("audio_b64")
    mode = (data.get("mode") or "talk").lower()
    duration = float(data.get("duration") or 4.0)
    fps = int(data.get("fps") or DEFAULT_FPS)

    if not image_b64:
        raise ValueError("image_b64 is required")
    if mode == "talk" and not audio_b64:
        raise ValueError("audio_b64 is required for mode talk")
    if not _sadtalker_available():
        raise RuntimeError("SadTalker belum terpasang di host ini")

    job = Path(tempfile.mkdtemp(prefix="job_", dir=str(WORKDIR)))
    img_path = _prep_image(base64.b64decode(image_b64), job / "src.png")
    out_dir = job / "out"

    if mode == "idle":
        try:
            if _liveportrait_available():
                vid = _render_liveportrait_idle(img_path, out_dir, duration, fps)
            else:
                raise RuntimeError("LivePortrait unavailable")
        except Exception as e:
            print("[avatar] idle -> STATIS tenang (LivePortrait tak tersedia:", e, ")", flush=True)
            vid = _render_static_idle(img_path, out_dir, fps, float(duration))
    else:
        aud_path = job / "drive.wav"
        aud_path.write_bytes(base64.b64decode(audio_b64))
        vid = _render_sadtalker(img_path, aud_path, out_dir, fps)

    final = job / "avatar.mp4"
    subprocess.run([
        "ffmpeg", "-y", "-i", str(vid),
        "-r", str(fps), "-pix_fmt", "yuv420p",
        "-movflags", "+faststart", "-c:v", "libx264", "-crf", "20",
        str(final),
    ], stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT, check=True)

    if bool(data.get("remove_bg")) and remove_background_video is not None:
        try:
            matted = remove_background_video(str(final), str(out_dir), int(fps))
            if matted and os.path.exists(str(matted)) and str(matted) != str(final):
                print("[avatar] background dihapus -> %s" % matted, flush=True)
                return Path(matted)
            print("[avatar] remove_bg tak menghasilkan webm, pakai mp4 biasa", flush=True)
        except Exception as e:
            print("[avatar] remove_bg gagal, fallback mp4:", e, flush=True)
    return final


# Registry job async avatar
_JOBS = {}
_JOBS_LOCK = threading.Lock()
_RENDER_LOCK = threading.Lock()  # serialisasi GPU (F5-TTS + SadTalker berbagi 1 T4)


def _synth(text, prompt_text, ref_wav_path, speed=0.78):
    if not text or not text.strip():
        raise ValueError("Teks input (text) kosong.")
    if not ref_wav_path or not os.path.exists(ref_wav_path) or os.path.getsize(ref_wav_path) < 100:
        raise ValueError("File sampel suara kosong atau tidak ditemukan oleh server F5-TTS.")

    ref_text = (prompt_text or "").strip()
    if not ref_text:
        ref_text = _transcribe_ref_audio(ref_wav_path)

    print(f"[F5-TTS] Voice cloning (ref: {os.path.getsize(ref_wav_path)}b, text: {len(text)} chars, speed: {speed:.2f}, nfe: 64)...", flush=True)
    with _RENDER_LOCK:
        try:
            res = F5_MODEL.infer(
                ref_file=ref_wav_path,
                ref_text=ref_text,
                gen_text=text.strip(),
                nfe_step=64,
                speed=float(speed),
                cfg_strength=2.0,
            )
        except TypeError:
            try:
                res = F5_MODEL.infer(
                    ref_file=ref_wav_path,
                    ref_text=ref_text,
                    gen_text=text.strip(),
                    nfe_step=64,
                    speed=float(speed),
                )
            except TypeError:
                res = F5_MODEL.infer(
                    ref_file=ref_wav_path,
                    ref_text=ref_text,
                    gen_text=text.strip(),
                )
        wav = res[0]
        sr = res[1]

    buf = io.BytesIO()
    sf.write(buf, wav, sr, format="WAV")
    print(f"[F5-TTS] SUKSES: Audio voice clone selesai digenerate (sample_rate={sr})!", flush=True)
    return buf.getvalue()


def _run_avatar_job(job_id, data):
    t0 = time.time()
    with _JOBS_LOCK:
        if job_id in _JOBS:
            _JOBS[job_id]["status"] = "running"
    try:
        with _RENDER_LOCK:
            final = _render_avatar_to_file(data)
        with _JOBS_LOCK:
            _JOBS[job_id].update(status="done", path=str(final))
        print("[avatar] job %s (%s) selesai %.1fs -> %s" % (job_id, data.get("mode"), time.time() - t0, final), flush=True)
    except Exception as e:
        traceback.print_exc()
        with _JOBS_LOCK:
            _JOBS[job_id].update(status="error", error=str(e))


class H(BaseHTTPRequestHandler):
    def log_message(self, *a):
        pass

    def _send_json(self, code, obj):
        body = json.dumps(obj).encode()
        self.send_response(code)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def do_GET(self):
        p = self.path.rstrip("/")
        if p.startswith("/avatar/result/"):
            job_id = p[len("/avatar/result/"):]
            with _JOBS_LOCK:
                job = dict(_JOBS.get(job_id) or {})
            if not job:
                self._send_json(404, {"status": "error", "error": "unknown job_id"})
                return
            st = job.get("status")
            if st in ("pending", "running"):
                self._send_json(202, {"status": st})
                return
            if st == "error":
                self._send_json(500, {"status": "error", "error": job.get("error") or "render failed"})
                return
            path = job.get("path")
            if not path or not os.path.exists(path):
                self._send_json(500, {"status": "error", "error": "result file missing"})
                return
            blob = Path(path).read_bytes()
            mime = "video/webm" if str(path).lower().endswith(".webm") else "video/mp4"
            self.send_response(200)
            self.send_header("Content-Type", mime)
            self.send_header("Content-Length", str(len(blob)))
            self.end_headers()
            self.wfile.write(blob)
            return
        if p == "/avatar/health":
            body = json.dumps({
                "status": "ok",
                "engine": "F5-TTS",
                "talk": _sadtalker_available(),
                "idle": _liveportrait_available() or _sadtalker_available(),
            }).encode()
            self.send_response(200)
            self.send_header("Content-Type", "application/json")
            self.end_headers()
            self.wfile.write(body)
            return
        if p in ["/health", "", "/tts", "/clone", "/avatar", "/lipsync"]:
            self.send_response(200); self.end_headers(); self.wfile.write(b"ok")
        else:
            self.send_response(404); self.end_headers()

    def do_POST(self):
        p = self.path.rstrip("/")
        try:
            n = int(self.headers.get("Content-Length", 0))
            data = json.loads(self.rfile.read(n) or b"{}")
        except Exception as e:
            self._send_json(400, {"error": f"bad json: {e}"})
            return

        if p in ["/avatar", "/lipsync"]:
            try:
                if not data.get("image_b64"):
                    raise ValueError("image_b64 is required")
                if (data.get("mode") or "talk").lower() == "talk" and not data.get("audio_b64"):
                    raise ValueError("audio_b64 is required for mode talk")
                job_id = uuid.uuid4().hex
                with _JOBS_LOCK:
                    _JOBS[job_id] = {"status": "pending", "path": None, "error": None}
                threading.Thread(target=_run_avatar_job, args=(job_id, data), daemon=True).start()
                print("[avatar] queued job", job_id, "mode", (data.get("mode") or "talk"), flush=True)
                self._send_json(202, {"job_id": job_id, "status": "pending"})
            except Exception as e:
                traceback.print_exc()
                self._send_json(400, {"error": str(e)})
            return

        if p in ["/tts", "/clone", ""]:
            try:
                text = data.get("text", "")
                prompt_text = data.get("prompt_text", "")
                speed = float(data.get("speed") or 0.78)
                ref_b64 = data.get("ref_audio_b64") or data.get("speaker_wav_b64")
                ref_path = None
                if ref_b64:
                    raw_tmp = tempfile.NamedTemporaryFile(suffix=".raw", delete=False)
                    raw_tmp.write(base64.b64decode(ref_b64))
                    raw_tmp.close()
                    clean_wav = tempfile.NamedTemporaryFile(suffix=".wav", delete=False).name
                    subprocess.run(["ffmpeg", "-y", "-i", raw_tmp.name, "-ar", "24000", "-ac", "1", clean_wav],
                                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
                    try: os.unlink(raw_tmp.name)
                    except Exception: pass
                    if os.path.exists(clean_wav) and os.path.getsize(clean_wav) > 100:
                        ref_path = clean_wav
                audio = _synth(text, prompt_text, ref_path, speed=speed)
                if ref_path and os.path.exists(ref_path):
                    try: os.unlink(ref_path)
                    except Exception: pass
                self.send_response(200)
                self.send_header("Content-Type", "audio/wav")
                self.send_header("Content-Length", str(len(audio)))
                self.end_headers()
                self.wfile.write(audio)
            except Exception as e:
                traceback.print_exc()
                msg = json.dumps({"error": str(e)}).encode()
                self.send_response(500)
                self.send_header("Content-Type", "application/json")
                self.end_headers()
                self.wfile.write(msg)
            return

        self.send_response(404); self.end_headers()


if __name__ == "__main__":
    print(f"[Pippit] Server F5-TTS + Avatar jalan di http://0.0.0.0:{PORT} (POST /tts [speed=0.78], POST /avatar ASYNC, GET /health)", flush=True)
    ThreadingHTTPServer(("0.0.0.0", PORT), H).serve_forever()


### 4) Jalankan server + URL publik gratis (cloudflared)
Biarkan sel ini TERUS berjalan. Muat model F5-TTS ~30 detik.

In [ ]:
import os, subprocess, time, re, urllib.request, http.client

os.environ.pop('PYTHONHASHSEED', None)
os.environ["F5_PORT"] = "8081"

# Cegah proses ganda di GPU
subprocess.run("pkill -9 -f pippit_server.py", shell=True)
subprocess.run("pkill -9 -f inference.py", shell=True)
time.sleep(2)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Unduh cloudflared (tunnel gratis, tanpa daftar)
if not os.path.exists("cloudflared"):
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "cloudflared")
    os.chmod("cloudflared", 0o755)

# Jalankan server di background
logf = open("server.log", "w")
srv = subprocess.Popen(["python", "-u", "/content/pippit_server.py"],
                       stdout=logf, stderr=subprocess.STDOUT)

print("Memuat model F5-TTS & menunggu server sehat (maks ~5 menit) ...")
ready = False
deadline = time.time() + 5*60
last = 0
while time.time() < deadline:
    if srv.poll() is not None:
        print("\n❌ Server BERHENTI (crash). Log:\n")
        print(open("server.log").read())
        raise SystemExit("Server gagal start — lihat traceback di atas.")
    try:
        with open("server.log") as f:
            f.seek(last); chunk = f.read(); last = f.tell()
        if chunk.strip():
            print(chunk, end="")
    except FileNotFoundError:
        pass
    try:
        c = http.client.HTTPConnection("127.0.0.1", 8081, timeout=3)
        c.request("GET", "/health"); r = c.getresponse()
        if r.status == 200:
            ready = True; print("\n✅ Server gabungan F5-TTS (speed=0.78) + Avatar SIAP."); break
    except Exception:
        pass
    time.sleep(4)

if not ready:
    print("\n⚠️ Server belum siap setelah 5 menit. Log:\n")
    print(open("server.log").read())
    raise SystemExit("Timeout saat memuat model F5-TTS.")

# Cek engine avatar
try:
    c = http.client.HTTPConnection("127.0.0.1", 8081, timeout=5)
    c.request("GET", "/avatar/health"); r = c.getresponse()
    print("[avatar/health]", r.read().decode())
except Exception as e:
    print("avatar health check gagal:", e)

# Buka tunnel ke port 8081 dan cetak URL
tun = subprocess.Popen(["./cloudflared","tunnel","--url","http://localhost:8081"],
                       stderr=subprocess.PIPE, text=True)
url = None
for line in tun.stderr:
    print(line.strip())
    m = re.search(r"https://[-\w]+\.trycloudflare\.com", line)
    if m:
        url = m.group(0)
        print("\n==============================================")
        print(">>> URL SERVER (F5-TTS Voice Clone + Talking Avatar):")
        print(">>>", url)
        print("    - PippitLokal: URL server VoxCPM/F5-TTS = URL di atas")
        print("    - PippitLokal: avatarColabUrl           = URL di atas (SAMA)")
        print("==============================================")
        break

print("\nBiarkan sel ini TERUS BERJALAN. Kalau Colab idle/putus, jalankan ulang sel ini.")


### Catatan Penting
- **Speed Dikalibrasi**: Default speed diset `0.78` agar tempo bicara rileks, artikulasi jelas, dan ada jeda alami.
- **Sampling 64 Step**: Artikulasi suara lebih tajam dibanding default 32 step.
- **Satu URL untuk keduanya**: `/tts` = F5-TTS voice clone, `/avatar` = video talking avatar.
